# Extract Skills from Job Descriptions (Multi-File Support)

This notebook processes multiple parquet files to avoid memory overflow.

**Features:**
- Handles hundreds of small parquet files
- Processes files one by one (low memory usage)
- Incremental saving (results saved after each file)
- Resume support (skip already processed files)
- Progress tracking

**Use Case:** Extract skills from 20M+ job descriptions split across multiple files.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob
from tqdm import tqdm
import json
import os
from skillner.jd_skill_extractor import JobDescriptionSkillExtractor

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

print("✓ Imports successful")

## Configuration

**IMPORTANT**: Change `INPUT_FOLDER` to point to your JD folder with multiple parquet files

In [ ]:
# =====================================================================
# EDIT THESE PATHS
# =====================================================================

# Input folder with multiple parquet files
INPUT_FOLDER = '../JD'  # Folder containing 200-300 small parquet files

# Knowledge base (created from ONET)
KB_PATH = '../.skillner-kb/MERGED_EN.pkl'  # Or ONET_EN.pkl

# Output
OUTPUT_PATH = '../data/jd_extracted_skills.parquet'
CHECKPOINT_FILE = '../data/processing_checkpoint.json'  # Track processed files

# Column names in your data
JD_TEXT_COLUMN = 'job_description'  # Column with job description text
ONET_CODE_COLUMN = 'onet_code'
DATE_COLUMN = 'post_date'

# Extraction parameters
SIMILARITY_THRESHOLD = 0.6  # Lower = more matches, higher = stricter
MAX_WINDOW_SIZE = 5  # Maximum words in a skill phrase

# Processing options
SAVE_EVERY_N_FILES = 1  # Save results after every N files (1 = after each file)
RESUME_FROM_CHECKPOINT = True  # Skip already processed files

# =====================================================================

print(f"Configuration:")
print(f"  Input folder: {INPUT_FOLDER}")
print(f"  KB: {KB_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Checkpoint: {CHECKPOINT_FILE}")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Max window size: {MAX_WINDOW_SIZE}")
print(f"  Resume from checkpoint: {RESUME_FROM_CHECKPOINT}")

## Step 1: Discover Input Files

In [ ]:
# Find all parquet files
input_files = sorted(glob(f"{INPUT_FOLDER}/*.parquet"))

print(f"Found {len(input_files)} parquet files in {INPUT_FOLDER}")

if len(input_files) == 0:
    print(f"\n⚠️ WARNING: No parquet files found in {INPUT_FOLDER}")
    print("Please check the INPUT_FOLDER path.")
else:
    print(f"\nFirst 5 files:")
    for f in input_files[:5]:
        file_size = os.path.getsize(f) / (1024 * 1024)  # MB
        print(f"  {os.path.basename(f):40s} ({file_size:.1f} MB)")
    
    if len(input_files) > 5:
        print(f"  ... and {len(input_files) - 5} more files")
    
    # Calculate total size
    total_size_gb = sum(os.path.getsize(f) for f in input_files) / (1024**3)
    print(f"\nTotal size: {total_size_gb:.2f} GB")

## Step 2: Check Checkpoint (Resume Support)

In [ ]:
# Load checkpoint if exists
processed_files = set()
if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint_data = json.load(f)
        processed_files = set(checkpoint_data.get('processed_files', []))
    print(f"✓ Loaded checkpoint: {len(processed_files)} files already processed")
else:
    print("Starting from scratch (no checkpoint found)")

# Filter out already processed files
files_to_process = [f for f in input_files if f not in processed_files]

print(f"\nFiles to process: {len(files_to_process)} / {len(input_files)}")
if len(processed_files) > 0:
    print(f"Skipping {len(processed_files)} already processed files")

## Step 3: Sample Check - Load One File

In [ ]:
# Load first file to check format
if len(files_to_process) > 0:
    sample_file = files_to_process[0]
    print(f"Loading sample file: {os.path.basename(sample_file)}")
    
    df_sample = pd.read_parquet(sample_file)
    
    print(f"\n✓ Loaded {len(df_sample):,} rows")
    print(f"\nColumns: {list(df_sample.columns)}")
    print(f"\nColumn check:")
    print(f"  {JD_TEXT_COLUMN}: {'✓ Found' if JD_TEXT_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
    print(f"  {ONET_CODE_COLUMN}: {'✓ Found' if ONET_CODE_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
    print(f"  {DATE_COLUMN}: {'✓ Found' if DATE_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
    
    print(f"\nSample data:")
    display(df_sample.head(3))
    
    # Check for null values
    null_count = df_sample[JD_TEXT_COLUMN].isna().sum()
    print(f"\nNull job descriptions: {null_count} / {len(df_sample)} ({null_count/len(df_sample)*100:.1f}%)")
else:
    print("No files to process!")

## Step 4: Initialize Skill Extractor

This loads the knowledge base and semantic model (may take 1-2 minutes):

In [ ]:
print("Initializing extractor...")
print("(This may take 1-2 minutes to load the semantic model)\n")

extractor = JobDescriptionSkillExtractor(
    kb_path=KB_PATH,
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_window_size=MAX_WINDOW_SIZE
)

print("\n✓ Extractor ready!")

## Step 5: Test on Single Example

Test extraction on one job description before processing all files:

In [ ]:
# Get first non-null job description from sample
if len(files_to_process) > 0:
    test_jd = df_sample[df_sample[JD_TEXT_COLUMN].notna()][JD_TEXT_COLUMN].iloc[0]
    
    print("Test Job Description:")
    print("=" * 70)
    print(test_jd[:500] + "..." if len(test_jd) > 500 else test_jd)
    print("=" * 70)
    
    # Extract skills
    print("\nExtracting skills...\n")
    result = extractor.extract_skills(test_jd, return_details=True)
    
    print(f"✓ Found {result['num_skills']} unique skills")
    print(f"\nSkills by category:")
    for section, skills in result['by_section'].items():
        print(f"  [{section}]: {len(skills)} skills")
        for skill in skills[:3]:
            print(f"    - {skill}")
        if len(skills) > 3:
            print(f"    ... and {len(skills) - 3} more")
    
    print(f"\nAll extracted skills:")
    for i, skill in enumerate(sorted(result['skills']), 1):
        print(f"{i:2d}. {skill}")

## Step 6: Process All Files

**This is the main processing loop.**

Processing strategy:
- Load one file at a time (low memory usage)
- Extract skills from all JDs in that file
- Append results to output file
- Update checkpoint
- Repeat for next file

**Estimated time:**
- ~2-5 seconds per job description
- If you have 100K JDs across all files: ~5-14 hours
- Progress is saved continuously, so you can stop and resume anytime

In [ ]:
print("="*70)
print("Starting Multi-File Processing")
print("="*70)
print(f"Files to process: {len(files_to_process)}")
print(f"Output: {OUTPUT_PATH}")
print(f"You can stop this cell anytime and resume later.\n")

# Track statistics
total_jds_processed = 0
total_files_processed = 0

# Process each file
for file_idx, file_path in enumerate(tqdm(files_to_process, desc="Processing files"), 1):
    file_name = os.path.basename(file_path)
    
    try:
        # Load file
        df_chunk = pd.read_parquet(file_path)
        
        # Skip if no job descriptions
        if JD_TEXT_COLUMN not in df_chunk.columns:
            print(f"\n⚠️ Skipping {file_name}: Column '{JD_TEXT_COLUMN}' not found")
            continue
        
        # Extract skills
        jd_list = df_chunk[JD_TEXT_COLUMN].tolist()
        results = extractor.extract_skills_batch(jd_list, show_progress=False)
        
        # Convert results to DataFrame
        results_df = pd.DataFrame([
            {
                'skills': r['skills'],
                'num_skills': r['num_skills'],
                'by_section': r['by_section']
            }
            for r in results
        ])
        
        # Combine with original data
        df_combined = pd.concat([df_chunk.reset_index(drop=True), results_df], axis=1)
        
        # Add quarter column if date available
        if DATE_COLUMN in df_combined.columns:
            df_combined[DATE_COLUMN] = pd.to_datetime(df_combined[DATE_COLUMN], errors='coerce')
            df_combined['quarter'] = df_combined[DATE_COLUMN].dt.to_period('Q')
        
        # Save results (append mode)
        if file_idx == 1 and not os.path.exists(OUTPUT_PATH):
            # First file: create new file
            df_combined.to_parquet(OUTPUT_PATH, index=False)
        else:
            # Subsequent files: append
            existing_df = pd.read_parquet(OUTPUT_PATH)
            combined_output = pd.concat([existing_df, df_combined], ignore_index=True)
            combined_output.to_parquet(OUTPUT_PATH, index=False)
        
        # Update checkpoint
        processed_files.add(file_path)
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({
                'processed_files': list(processed_files),
                'total_files': len(input_files),
                'files_remaining': len(input_files) - len(processed_files)
            }, f, indent=2)
        
        # Update statistics
        total_jds_processed += len(df_chunk)
        total_files_processed += 1
        
        # Print progress every 10 files
        if file_idx % 10 == 0:
            print(f"\n✓ Processed {file_idx}/{len(files_to_process)} files ({total_jds_processed:,} JDs)")
        
    except Exception as e:
        print(f"\n✗ Error processing {file_name}: {e}")
        continue

print("\n" + "="*70)
print("PROCESSING COMPLETE")
print("="*70)
print(f"Files processed: {total_files_processed}")
print(f"Job descriptions processed: {total_jds_processed:,}")
print(f"Output saved to: {OUTPUT_PATH}")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")

## Step 7: Load Final Results and Generate Statistics

In [ ]:
# Load final combined results
print("Loading final results...")
df_final = pd.read_parquet(OUTPUT_PATH)

print(f"✓ Loaded {len(df_final):,} job descriptions")
print(f"\nColumns: {list(df_final.columns)}")
print(f"\nMemory usage: {df_final.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

# Basic statistics
print("\n" + "="*70)
print("Basic Statistics")
print("="*70)
print(f"Total job descriptions: {len(df_final):,}")
print(f"\nSkills per job description:")
print(f"  Mean: {df_final['num_skills'].mean():.1f}")
print(f"  Median: {df_final['num_skills'].median():.1f}")
print(f"  Min: {df_final['num_skills'].min():.0f}")
print(f"  Max: {df_final['num_skills'].max():.0f}")
print(f"  Std: {df_final['num_skills'].std():.1f}")

# Show sample
print(f"\nSample results:")
display(df_final[['onet_code', 'num_skills']].head(10))

## Step 8: Detailed Statistics

In [ ]:
# Count skills by section
section_counts = {}
all_skills = []

for idx, row in df_final.iterrows():
    all_skills.extend(row['skills'])
    for section, skills in row['by_section'].items():
        section_counts[section] = section_counts.get(section, 0) + len(skills)

print("="*70)
print("Skills by KSAO Category")
print("="*70)
for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {section:30s}: {count:8,} skills")

# Most common skills
from collections import Counter
skill_freq = Counter(all_skills)

print(f"\n{'='*70}")
print(f"Top 20 Most Common Skills")
print("="*70)
for i, (skill, count) in enumerate(skill_freq.most_common(20), 1):
    pct = (count / len(df_final)) * 100
    print(f"{i:2d}. {skill:45s} ({count:6,} times, {pct:5.1f}%)")

print(f"\nTotal unique skills found: {len(skill_freq):,}")

## Step 9: Visualizations

In [ ]:
# Distribution of skills per JD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_final['num_skills'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Skills per Job Description')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Skills per JD')
axes[0].axvline(df_final['num_skills'].mean(), color='red', linestyle='--', 
               label=f"Mean: {df_final['num_skills'].mean():.1f}")
axes[0].legend()

# Box plot by ONET code (top 10)
if ONET_CODE_COLUMN in df_final.columns:
    top_onet = df_final[ONET_CODE_COLUMN].value_counts().head(10).index
    df_top = df_final[df_final[ONET_CODE_COLUMN].isin(top_onet)]
    df_top.boxplot(column='num_skills', by=ONET_CODE_COLUMN, ax=axes[1], rot=45)
    axes[1].set_xlabel('ONET Code')
    axes[1].set_ylabel('Number of Skills')
    axes[1].set_title('Skills per JD by ONET Code (Top 10)')
    plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Skills by category
plt.figure(figsize=(12, 6))
sections = list(section_counts.keys())
counts = list(section_counts.values())
plt.bar(sections, counts, edgecolor='black', alpha=0.7)
plt.xlabel('KSAO Section')
plt.ylabel('Total Skills Extracted')
plt.title('Skills Extracted by KSAO Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 10: Time Series Analysis (Optional)

In [ ]:
if 'quarter' in df_final.columns:
    # Average skills per quarter
    quarterly_stats = df_final.groupby('quarter')['num_skills'].agg(['mean', 'median', 'count'])
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Skills over time
    axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['mean'], 
                marker='o', label='Mean', linewidth=2)
    axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['median'], 
                marker='s', label='Median', linewidth=2)
    axes[0].set_xlabel('Quarter')
    axes[0].set_ylabel('Skills per JD')
    axes[0].set_title('Average Skills per Job Description Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Number of JDs per quarter
    axes[1].bar(quarterly_stats.index.astype(str), quarterly_stats['count'], 
               edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Quarter')
    axes[1].set_ylabel('Number of Job Descriptions')
    axes[1].set_title('Job Descriptions per Quarter')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\nQuarterly statistics:")
    display(quarterly_stats)
else:
    print("No date information available for time series analysis")

## Step 11: Save Summary Statistics

In [ ]:
# Save statistics as JSON
stats_path = OUTPUT_PATH.replace('.parquet', '_stats.json')

stats_summary = {
    'total_jds': len(df_final),
    'total_files_processed': len(processed_files),
    'unique_skills_total': len(skill_freq),
    'skills_per_jd': {
        'mean': float(df_final['num_skills'].mean()),
        'median': float(df_final['num_skills'].median()),
        'min': int(df_final['num_skills'].min()),
        'max': int(df_final['num_skills'].max()),
        'std': float(df_final['num_skills'].std())
    },
    'by_section': {k: int(v) for k, v in section_counts.items()},
    'top_20_skills': [[skill, int(count)] for skill, count in skill_freq.most_common(20)]
}

with open(stats_path, 'w') as f:
    json.dump(stats_summary, f, indent=2)

print(f"✓ Statistics saved to: {stats_path}")

print("\n" + "="*70)
print("ALL PROCESSING COMPLETE")
print("="*70)
print(f"Processed: {len(df_final):,} job descriptions from {len(processed_files)} files")
print(f"Extracted: {len(skill_freq):,} unique skills")
print(f"Output: {OUTPUT_PATH}")
print(f"Statistics: {stats_path}")
print(f"Checkpoint: {CHECKPOINT_FILE}")

## Clean Up (Optional)

After successful completion, you can delete the checkpoint file:

In [ ]:
# Uncomment to delete checkpoint file
# import os
# if os.path.exists(CHECKPOINT_FILE):
#     os.remove(CHECKPOINT_FILE)
#     print(f"✓ Deleted checkpoint file: {CHECKPOINT_FILE}")

## Next Steps

Now that you have extracted skills from all files:

1. **Temporal Analysis**: Use `notebooks/jd_temporal_analysis.ipynb` for time series analysis
2. **Occupation Analysis**: Filter by `onet_code` to compare different occupations
3. **KSAO Trends**: Analyze how different skill categories change over time
4. **Emerging Skills**: Identify skills that are increasing in frequency

**Tips:**
- The output file contains all results in a single parquet file
- If you need to reprocess: delete checkpoint file and output file, then rerun
- You can process different subsets by adjusting INPUT_FOLDER